# Objective: #
Create a pipeline to go from the MSC dataset to a clean ROI timeseries.

In [17]:
# !fmriprep-docker --fs-license-file /home/tahia/Documents/MSC/scripts/license.txt /home/tahia/Documents/MSC/data/ds000224 /home/tahia/Documents/MSC/data/fmri_output/derivatives participant -w /home/tahia/Documents/MSC/data/fmri_output/scratch --task-id memoryscenes --skip-bids-validation

In [18]:
import os
import sys
import logging
import warnings
from datetime import datetime
import json
import numpy as np
import pandas as pd
import nibabel as nib
import nilearn as nil
from nilearn.plotting import plot_epi
from nilearn.glm.first_level import make_first_level_design_matrix

In [19]:
# Set up logging and deal with any warnings.
today = datetime.now().date().strftime("%Y-%m-%d")

logging.basicConfig(
    level=logging.INFO, 
    format="%(asctime)s - %(levelname)s - %(message)s", 
    datefmt="%Y-%m-%d %H:%M:%S", 
    handlers=[
        logging.FileHandler(f"./logs/msc_data_generation_{today}.log", mode='w'), 
        logging.StreamHandler(sys.stdout)
    ]
)

warnings.filterwarnings('ignore')

In [58]:
# A dictionary of all the stuff we'd otherwise have to keep writing out; 
# this way, we only ever have to change things up here.
config = {
    'data_dir': "../data/fmriprep_output/derivatives",
    'raw_dir': "../data/ds000224",
    'roi_mask_suffix': "Sphere_Mask.nii",
    'subj_prefix': "sub-MSC",
    'sess_prefix': "ses-func",
    'task_label': "task-memoryscenes",
    'space_label': "space-MNI152NLin2009cAsym",
    'bold_suffix': "desc-preproc_bold.nii.gz",
    'mask_suffix': "desc-brain_mask.nii.gz",
    'confounds_suffix': "desc-confounds_timeseries.tsv",
    'events_suffix': 'events.tsv',
    'data_output_dir': 'datasets', 
    'scan_dim_3d': (0,0,0),
    'roi_label': 'roi-rnd_r',
    'roi_coord_MNI': np.random.randint(low=-24,high=24,size=3), #(24, -2, -20),
    'roi_radius': 60,
    'encoding_label': 'encoding-stimxhrf',
    'csv_suffix': 'desc-vox_w_stim_hrf.csv',
    'nilearn_defaults': {
         'standardize': 'psc',
         'detrend': True,
         'low_pass': 0.1,
         'high_pass': 0.01,
         't_r': 2
        },
    'hrf_model': 'spm'
}

In [59]:
def json_compatibility_encoder(obj):
    """
    A helper function to convert numpy types to python floats  
    and ints to keep the json library from getting mad about 
    type unserializability. If we can't convert to the right 
    numeric type, we go for string type.

    Parameters
    ----------
    obj : numeric, string, or other data types
        Meta data from the global config variable that need
        to be converted into native python types.

    Returns
    -------
    float, int, string
        Converts object into the correct type for the json file.
    """
    if isinstance(obj, (np.floating, np.float32, np.float64)):
        return float(obj)
    if isinstance(obj, (np.integer, np.int32, np.int64)):
        return int(obj)
    return str(obj)

In [60]:
def save_dataset_with_metadata(df):
    """
    Saves the dataframe and its metadata to a unique, programmatically named directory
    to prevent dataset over-writing and to ensure dataset generation traceability.

    Parameters
    ----------
    df : pandas.DataFrame
        Fully processed and concatenated pandas dataframe containing timeseries for voxels in ROI.
    config : dictionary
        Dictionary containing all parameters for dataset generation
    """
    gen_id = f"dataset_{config['task_label']}_{config['roi_label']}"
    
    gen_dir = os.path.join(config['data_output_dir'], gen_id)
    os.makedirs(gen_dir, exist_ok=True)
    
    csv_path = os.path.join(gen_dir, "_".join([gen_id, str(config['roi_coord_MNI']), f"date-{today}", config['csv_suffix']]))
    json_path = os.path.join(gen_dir, "_".join([gen_id, str(config['roi_coord_MNI']), f"date-{today}", "metadata.json"]))
    
    metadata = {
        "generation_id": gen_id,
        "generation_date": today,
        "config": config,
        "data_summary": {
            "total_rows": len(df),
            "subjects": df['subj'].unique().tolist() if 'subj' in df.columns else "N/A",
            "sessions": df['sess'].unique().tolist() if 'sess' in df.columns else "N/A",
            "columns": df.columns.tolist()
        }
    }
    
    df.to_csv(csv_path, index=False)
    with open(json_path, 'w') as f:
        json.dump(metadata, f, indent=4, default=json_compatibility_encoder)
        
    logging.info("Successfully saved unique dataset and metadata to: %s", gen_dir)

In [61]:
# Sphere mask creation
# Turns out this is more annoying than expected, so let's just do math. 

def ROI_sphere(path_to_brain, MNI_coord, radius):
    """
    Returns an ROI sphere mask for a 3d or 4d volume given the center and radius.

    Parameters
    ----------
    path_to_brain : str
        File path to the 3D or 4D NIfTI image containing the brain image.
    MNI_coord : tuple
        The x, y, z coordinates for the center of the sphere in MNI space.
    radius : float
        The radius of the sphere in millimeters.

    Returns
    -------
    nibabel.nifti1.Nifti1Image
        A mask ready to be used with the input image.

    Raises
    ------
    FileNotFoundError, nib.filebasedimages.ImageFileError
        If either the file specified by the given path is missing or if the file 
        cannot be read by nibabel for whatever reason. 
    Exception
        If other unexpected errors arise.
    """
    logging.info("Initiating ROI mask creation for coordinate: %s.", MNI_coord)
    using_template = False

    # Open brain file
    try:
        nib_img = nib.load(path_to_brain)
    except (FileNotFoundError, nib.filebasedimages.ImageFileError):
        # In the event of missing brain, we log the issue and open
        # an MNI template because the specific brain doesn't matter for 
        # ROI creation as they should all be in MNI space.
        logging.warning("Brain image missing or invalid at location: %s. \nSwitching to template...", os.path.abspath(path_to_brain))
        nib_img = nil.datasets.load_mni152_template()
        using_template = True
    except Exception as e:
        logging.error("Unexpected failure %s when loading nifti image file %s or when loading template brain image.", 
                      e, os.path.abspath(path_to_brain), exc_info=True)
        
        
    # First, we need to change the center of our ROI from MNI space to voxel space.
    inverse_aff = np.linalg.inv(nib_img.affine)
    MNI_homo = [c for c in MNI_coord]
    MNI_homo.append(1)
    vox_center_homo = inverse_aff @ MNI_homo
    i,j,k = vox_center_homo[:3].astype(int)

    # We also need an index grid.  Now, we don't know whether we will get 3d or 4d data, 
    # so we'll have to just take the first three dimensions.
    (coordi, coordj, coordk) = np.indices(nib_img.shape[:3])

    # Now we need to transfer the radius into voxel space as well.  For this, we need the 
    # spatial resolution of the scan; basically, a vox = ? mm
    if using_template:
        dimx,dimy,dimz = config['scan_dim_3d']
    else:
        dimx,dimy,dimz = nib_img.header.get_zooms()[:3]
    assert dimx==dimy and dimy==dimz, "The dimensions of the voxel are not the same size."
    vox_rad = (radius + (dimx/2))//dimx # I could've rounded and converted to int but I think this might be slightly more efficient.
    
    # Now, we create the cube in which our sphere is circumscribed. We are naming the start
    # and stop coordinates using 0 and 1 respectively because, well, it's short and makes sense.
    i0, i1 = int(i-vox_rad), int(i+vox_rad)
    j0, j1 = int(j-vox_rad), int(j+vox_rad)
    k0, k1 = int(k-vox_rad), int(k+vox_rad)

    # We can now use these indices to subset our grid
    bbi = coordi[i0:i1, j0:j1, k0:k1]
    bbj = coordj[i0:i1, j0:j1, k0:k1]
    bbk = coordk[i0:i1, j0:j1, k0:k1]

    # Now let's get the euclidean distances between the center and all other voxels
    eu_dist_bb = np.sqrt(((bbi-i)**2)+((bbj-j)**2)+((bbk-k)**2))
    maskbb = (eu_dist_bb <= vox_rad)

    # We have the sphere!  But it's in the bounding box.  Let's move it to where it's supposed to be.
    mask_img_data = np.zeros(nib_img.shape[:3], dtype=float)
    mask_img_data[i0:i1, j0:j1, k0:k1] = maskbb
    
    # Now, after all that, we return the mask as a 3d nib image.
    mask_img = nib.Nifti1Image(mask_img_data, nib_img.affine)

    return mask_img

In [62]:
def skull_strip(path_to_brain, path_to_mask):
    """
    Applies the brain mask from the specified path to the preprocessed brain from the
    specified path. 
    
    Parameters
    ----------
    path_to_brain : str
        File path to the 4D NIfTI image containing the brain and skull labeled 
        as the preproc_brain by fMRIPrep.
    path_to_mask : str
        File path to the binary 3D NIfTI brain mask also generated by fMRIPrep.

    Returns
    -------
    nibabel.nifti1.Nifti1Image or None
        A new 4D NIfTI image object containing the skull-stripped brain data,
        preserving the original affine and header if possible. If skull-stripping
        isn't possible, the original 4D NIfTI image is loaded and returned. 
        If the 4D NIfTI image cannot be found or opened, None is returned.

    Raises
    ------
    FileNotFoundError, nib.filebasedimages.ImageFileError
        If either the file specified by the given path is missing or if the file 
        cannot be read by nibabel for whatever reason.
    ValueError
        If either the values within a nibabel-loaded NIfTI image are incorrect or 
        if the shape of the image is incorrect.
    Exception
        If other unexpected errors arise.
    """
    
    logging.info("Initiating skull-stripping for pre-processed brain %s.", os.path.basename(path_to_brain))

    # Open brain file
    try:
        brain_img = nib.load(path_to_brain)
    except (FileNotFoundError, nib.filebasedimages.ImageFileError):
        # In the event of missing brain, we log the issue and return
        # None so that the main script is aware this is a big nada
        logging.warning("Brain image missing or invalid at location: %s. \nSkipping run...", os.path.abspath(path_to_brain))
        return None
    except Exception as e:
        logging.error("Unexpected failure %s when loading nifti image file %s.", e, os.path.abspath(path_to_brain), exc_info=True)
        raise
        
    # Open mask file; return original brain image on failure
    try:
        mask_img = nib.load(path_to_mask)
    except (FileNotFoundError, nib.filebasedimages.ImageFileError):
        # In the event of missing mask, we log the issue and return
        # the original brain image we loaded so that the script can
        # continue even if skull-stripping fails. As the brains are
        # all in the same space, this may not end up being an issue.
        logging.warning("Mask image missing or invalid at location: %s. \nSkipping skull-stripping...", os.path.abspath(path_to_mask))
        return brain_img
    except Exception as e:
        logging.error("Unexpected failure %s when opening file %s.", e, os.path.abspath(path_to_mask), exc_info=True)
        return brain_img

    # Check if mask image if binary
    mask_vals = mask_img.get_fdata()
    binary_test = np.all(np.isclose(mask_vals, 0, atol=1e-5) | np.isclose(mask_vals, 1, atol=1e-5))
    if not binary_test:
        # If both brain image and mask image are proper nifti files that opened without problems,
        # if the mask is not in face binary, we are having a HUGE problem somewhere: is everything named
        # correctly? did fMRIPrep fail somehow? did the user put in the wrong path? is the brain image 
        # file even the correct one?! This is thus an epic error.
        logging.critical("Type Mismatch: Non-binary nifti image file found in location: %s when looking for brain mask!", os.path.abspath(path_to_mask))
        raise ValueError(f"Fatal Error: Binary nifti image file expected at location {os.path.abspath(path_to_mask)}; got non-binary nifti image file instead!")

    # Actually stripping the skull
    try:
        skull_stripped_img = brain_img.get_fdata() * mask_vals[..., None]
    except ValueError:
        logging.critical("Shape Mismatch: The brain %s (shape: %s) and the mask %s (shape: %s) are in different spaces.", os.path.basename(path_to_brain), brain_img.shape, os.path.basename(path_to_mask), mask_img.shape, exc_info=True)
        raise
    except Exception as e:
        # This should not happen.  At this point, the only explanation is something is wrong with the brain image somehow. 
        logging.error("Unexpected failure when applying mask to brain image %s: %s. \nReturning original brain image...", os.path.basename(path_to_brain), e, exc_info=True)
        return brain_img

    # There should be no problem here
    preproc_bold = nib.Nifti1Image(skull_stripped_img, brain_img.affine, brain_img.header)
    logging.info("Successfully skull-stripped %s!", os.path.basename(path_to_brain))
    return preproc_bold

In [63]:
def brain_to_tsdf(brain_img, path_to_ROI_mask, path_to_confounds_file, save_path=None, **nil_params):
    """
    Creates a pandas dataframe in the form of time x voxels using a 4D NIfTI file and 
    its Region of Interest mask. It also optionally takes in a confounds file (an output
    of fMRIPrep) to further clean the dataset.

    Parameters
    ----------
    brain_img: nibabel.nifti1.Nifti1Image
        A 4D image file loaded into memory through nibabel.
    path_to_ROI_mask: str
        A string containing the path to the nii or nii.gz file containing the binary mask for 
        the region of interest.
    path_to_confounds_file: str
        A string containing the path to the confounds file (an output of fMRIPrep) for the
        subject and run to which the brain image (brain_img) belongs.
    save_path: str or None
        A string containing the path to where the time series dataset ought be saved as CSV.
    **nil_params: parameter name and value
        Named nilearn.maskers.NiftiMasker parameters with their values for passing on to nilearn.

    Returns
    -------
    pandas.DataFrame
        The timeseries for each voxel within the region of interest is extracted, cleaned (detrended, 
        filtered, etc.), and put into a dataframe (n_scans x n_voxels) for future analysis.

    Raises
    ------
    FileNotFoundError, nib.filebasedimages.ImageFileError
        If either the file specified by the given path is missing or if the file 
        cannot be read by nibabel for whatever reason.
    Exception
        If other unexpected errors arise.
    """
    # identifier = os.path.basename(path_to_confounds_file).split("_")
    
    # logging.info("Initiating ROI time series data extraction for %s, %s.", identifier[0], identifier[1])
    logging.info("Initiating ROI time series data extraction...")

    # We first load the ROI. Without the ROI mask, we can't do anything for this run.
    try:
        sphere_mask = nib.load(path_to_ROI_mask)
    except (FileNotFoundError, nib.filebasedimages.ImageFileError):
        logging.error("The ROI mask is missing or invalid at location: %s!", path_to_ROI_mask)
        raise
    except Exception as e:
        logging.error("Unexpected exception occurred: %s.", e, exc_info=True)
        raise 

    # Next, we load the confounds file if possible.
    try:
        cfdf = pd.read_csv(path_to_confounds_file, header=0, sep="\t")
        cfdf = cfdf.fillna(0)
    except FileNotFoundError:
        logging.warning("No confounds file found; continuing without cleaning...")
        cfdf = None
    except Exception as e:
        logging.error("Unexpected exception occurred: %s; continuing without cleaning...", e, exc_info=True)
        cfdf = None

    # We update the default params and add in other params as specified by the user
    default_nil_params = {'standardize':'zscore_sample', 
                          'standardize_confounds':True, 
                          'detrend':True, 
                          'high_variance_confounds':False, 
                          'low_pass':0.1, 
                          'high_pass':0.01, 
                          't_r':2, 
                          }
    maskparams = default_nil_params | config['nilearn_defaults'] | nil_params
    config['nilearn_defaults'] = maskparams # update config as well so that it has the right info.
    
    FEF_mask = nil.maskers.NiftiMasker(sphere_mask, **maskparams)

    # Nilearn is planning on implementing a method that allows niftimasker to directly output a df
    # This try/except block is to use it if available but use the helper function if not.
    # the method in question is .set_output()
    set_output_available = False
    
    try:
        FEF_mask.set_output(transform='pandas')
        set_output_available = True
    except (NotImplementedError, AttributeError):
        logging.info("nil.maskers.NiftiMasker.set_output() is currently unavailable. Using manual method instead.")
    
    FEF_ts = FEF_mask.fit_transform(brain_img, confounds=cfdf)

    if set_output_available:
        FEF_ts['scan_times'] = np.round(np.arange(brain_img.shape[3])*maskparams['t_r'], 2)
        FEF_ts = FEF_ts.set_index('scan_times')
    else:
        FEF_ts = ts_mat_to_df(FEF_ts, t_r=maskparams['t_r'])
    
    if save_path is not None:
        try:
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            FEF_ts.to_csv(save_path)
        except Exception as e:
            logging.error("Failed to save time series dataframe to %s due to error: %s!", save_path, e, exc_info=True)
    return FEF_ts

In [64]:
def ts_mat_to_df(ts_mat, t_r=2.2):
    """
    Takes in a number of scans x number of voxels array and returns a dataframe.

    Parameters
    ----------
    ts_mat: numpy.ndarray
        An n_scans x n_voxels array of a 4D brain wherein n_voxels refers to the flattened 
        brain and n_scans refers to each time the brain was scanned.
    t_r: float
        The time interval between scans in seconds.

    Returns
    -------
    pandas.DataFrame
        A dataframe of the same shape as the input ts_mat with columns and index labeled.

    TODO: remove once no longer in use; i.e., nil implements set_output().
    """
    
    idx = np.round(np.arange(ts_mat.shape[0])*t_r, 2)
    roidf = pd.DataFrame(data=ts_mat, index=idx, columns=["v"+str(i) for i in range(ts_mat.shape[1])])
    
    return roidf

In [65]:
def design_matrix_creation(path_to_events_file, ts_index, **nil_params):
    """
    Creates the design matrix using the stimulus onset file that is included with fMRI scans.

    Parameters
    ----------
    path_to_events_file: str
        A string containing the path to the tab-separated-file (.tsv) that contains stimulus
        onset times, stimulus types, and durations for the current subject and session.
    ts_index: list of floats or pandas.Series of floats
        A list or pandas.Series of the scan times for the associated brain scan; this 
        information can be found as the index of the ROI dataframe outputted by the brain_to_tsdf
        function or by creating a list of multiples of the TR with a length matching the number 
        scans for the associated BOLD scan.
    **nil_params: parameter name and value, optional
        Named nilearn.glm.FirstLevel.make_first_level_design_matrix parameters with their values 
        for passing on to nilearn.

    Returns
    -------
    pandas.DataFrame
        The design matrix formed using the stimulus onset file and the specified hemodynamic 
        response function, along with other named parameters.

    Raises
    ------
    FileNotFoundError
        If the user-specified path does not lead to a tab-separated-file.
    KeyError
        If 'trial_type', 'onset', or 'duration' is not one of the column names in the events file.
    Exception
        If unexpected errors arise.
    """
    logging.info("Creating design matrix from events file...")
    
    try:
        eventsdf = pd.read_csv(path_to_events_file, header=0, sep="\t")
    except FileNotFoundError:
        logging.error("Failure to locate event file at %s!", path_to_events_file, exc_info=True)
        raise
    except Exception as e:
        logging.error("Unexpected error when trying to read file %s into a dataframe: %s.", path_to_events_file, e, exc_info=True)
        raise

    try:
        eventsdf = eventsdf[['onset','duration','trial_type']] # making sure we only have the cols we need
    except KeyError:
        logging.error("Events file successfully found at %s but does not have correct columns!", path_to_events_file, exc_info=True)
        raise
    
    try:
        eventsdf['trial_type'] = eventsdf['trial_type'].map(lambda x: x.split("_")[0]) # Format column to simplify analysis
    except Exception as e:
        logging.error("Unknown error occurred when trying to format the trial_type column in the events dataframe: %s", e, exc_info=True)
        raise

    # These are default parameters from nilearn's make_first_level_design_matrix function
    default_nil_params = {'hrf_model':'glover', 
                          'drift_model':'cosine', 
                          'high_pass':0.01, 
                          'drift_order':1, 
                          'fir_delays':None, 
                          'add_regs':None, 
                          'add_reg_names':None, 
                          'min_onset':-24, 
                          'oversampling':50
                          }
    paramsdict = default_nil_params | nil_params
    
    des_mat = make_first_level_design_matrix(ts_index, eventsdf, **paramsdict)
    return des_mat

In [66]:
for sub in range(1,11):
    if sub==10:
        sub = str(sub)
    else:
        sub = "0"+str(sub)
    
    for ses in range(1,15):
        if ses>9:
            ses = str(ses)
        else:
            ses = "0"+str(ses)
        
        p_brain = os.path.join(config['data_dir'], 
                               config['subj_prefix']+sub, 
                               config['sess_prefix']+ses, 
                               'func', 
                               "_".join([config['subj_prefix']+sub, 
                                         config['sess_prefix']+ses, 
                                         config['task_label'], 
                                         config['space_label'], 
                                         config['bold_suffix']
                                        ]
                                       )
                              )
        
        p_mask = os.path.join(config['data_dir'], 
                              config['subj_prefix']+sub, 
                              config['sess_prefix']+ses, 
                              'func', 
                              "_".join([config['subj_prefix']+sub, 
                                        config['sess_prefix']+ses, 
                                        config['task_label'], 
                                        config['space_label'], 
                                        config['mask_suffix']
                                       ]
                                      )
                             )
        
        p_sphere = os.path.join(config['data_output_dir'], 
                                "_".join([config['subj_prefix']+sub, 
                                          config['sess_prefix']+ses, 
                                          config['roi_label'], 
                                          str(config['roi_coord_MNI']), 
                                          config['roi_mask_suffix']
                                         ]
                                        )
                               )
        
        p_confounds = os.path.join(config['data_dir'], 
                                   config['subj_prefix']+sub, 
                                   config['sess_prefix']+ses, 
                                   'func', 
                                   "_".join([config['subj_prefix']+sub, 
                                             config['sess_prefix']+ses, 
                                             config['task_label'], 
                                             config['confounds_suffix']
                                            ]
                                           )
                                  )
        
        p_events = os.path.join(config['raw_dir'], 
                                config['subj_prefix']+sub, 
                                config['sess_prefix']+ses, 
                                'func',  
                                "_".join([config['subj_prefix']+sub, 
                                          config['sess_prefix']+ses, 
                                          config['task_label'], 
                                          config['events_suffix']
                                         ]
                                        )
                               )
        
        if not os.path.isfile(p_brain):
            print(f"Subject {sub}, session {ses} failed.")

            continue

        # get bold and get some params
        skull_stripped_save_path = os.path.join(config['data_dir'], 
                                                config['subj_prefix']+sub, 
                                                config['sess_prefix']+ses, 
                                                'func', 
                                                "_".join([config['subj_prefix']+sub, 
                                                          config['sess_prefix']+ses, 
                                                          config['task_label'], 
                                                          config['space_label'], 
                                                          'desc-skull_stripped_preproc_bold.nii.gz'
                                                          ]
                                                         )
                                                )
        bold_img = None
        bold_img_is_new = False
        try: 
            bold_img = nib.load(skull_stripped_save_path)
            logging.info("Found skull-stripped BOLD image on disk for %s, %s. Loading...", 
                         config['subj_prefix']+sub, config['sess_prefix']+ses)
        except (FileNotFoundError, nib.filebasedimages.ImageFileError):
            logging.info("File not found. Stripping skull...")
            bold_img = skull_strip(p_brain, p_mask)
            if bold_img is not None:
                nib.save(bold_img, skull_stripped_save_path)

        if os.path.isfile(skull_stripped_save_path):
            logging.info("Already skull stripped!")
            bold_img = nib.load(skull_stripped_save_path)
        else:
            bold_img = skull_strip(p_brain, p_mask)
            nib.save(bold_img, skull_stripped_save_path)

        vx,vy,vz,tr_val = bold_img.header.get_zooms()
        config['scan_dim_3D'] = (vx,vy,vz)

        # get the timeseries for the ROI
        if not os.path.isfile(p_sphere):
            roi_mask = ROI_sphere(p_brain, config['roi_coord_MNI'], config['roi_radius'])
            nib.save(roi_mask, p_sphere)
        else:
            logging.info("ROI sphere mask already exists!")
            
        fef_ts = brain_to_tsdf(bold_img, p_sphere, p_confounds, t_r=tr_val)
        fdf = ts_mat_to_df(fef_ts, t_r=tr_val)
        dmdf = design_matrix_creation(p_events, fdf.index)

        fdf['indoor'] = dmdf['indoor']
        fdf['outdoor'] = dmdf['outdoor']
        
        fdf.to_csv(os.path.join(config['data_dir'], 
                                config['subj_prefix']+sub, 
                                config['sess_prefix']+ses, 
                                'datasets', 
                                "_".join([config['subj_prefix']+sub, 
                                          config['sess_prefix']+ses, 
                                          config['task_label'], 
                                          config['roi_label'], 
                                          config['encoding_label'], 
                                          config['csv_suffix']
                                         ]
                                        )
                               )
                  )

2026-01-02 20:51:38 - INFO - Found skull-stripped BOLD image on disk for sub-MSC01, ses-func01. Loading...
2026-01-02 20:51:38 - INFO - Already skull stripped!
2026-01-02 20:51:38 - INFO - Initiating ROI mask creation for coordinate: [-4 -8 21].
2026-01-02 20:51:38 - INFO - Initiating ROI time series data extraction...
2026-01-02 20:51:38 - INFO - nil.maskers.NiftiMasker.set_output() is currently unavailable. Using manual method instead.
2026-01-02 20:51:41 - INFO - Creating design matrix from events file...
2026-01-02 20:51:42 - INFO - Found skull-stripped BOLD image on disk for sub-MSC01, ses-func02. Loading...
2026-01-02 20:51:42 - INFO - Already skull stripped!
2026-01-02 20:51:42 - INFO - Initiating ROI mask creation for coordinate: [-4 -8 21].
2026-01-02 20:51:42 - INFO - Initiating ROI time series data extraction...
2026-01-02 20:51:42 - INFO - nil.maskers.NiftiMasker.set_output() is currently unavailable. Using manual method instead.
2026-01-02 20:51:44 - INFO - Creating design

In [67]:
df_list = []
for sub in range(1,11):
    if sub==10:
        sub = str(sub)
    else:
        sub = "0"+str(sub)
    
    for ses in range(1,15):
        if ses>9:
            ses = str(ses)
        else:
            ses = "0"+str(ses)

        p_fdf = os.path.join(config['data_dir'], 
                             config['subj_prefix']+sub, 
                             config['sess_prefix']+ses, 
                             'datasets', 
                             "_".join([config['subj_prefix']+sub, 
                                       config['sess_prefix']+ses, 
                                       config['task_label'], 
                                       config['roi_label'], 
                                       config['encoding_label'], 
                                       config['csv_suffix']
                                      ]
                                     )
                            )

        if not os.path.isfile(p_fdf):
            print(f"Failed: {p_fdf}")
            continue

        df = pd.read_csv(p_fdf, header=0)
        df['sess'] = [ses]*df.shape[0]
        df['subj'] = [sub]*df.shape[0]
        df['run'] = len(df_list)+1
        df_list.append(df)
        print(f"Subject {sub}, session {ses}, accumulated dfs {len(df_list)}")

fdf = pd.concat(df_list, axis=0, ignore_index=True)
fdf = fdf.rename(columns={"Unnamed: 0":"scan_time"})

Subject 01, session 01, accumulated dfs 1
Subject 01, session 02, accumulated dfs 2
Subject 01, session 03, accumulated dfs 3
Subject 01, session 04, accumulated dfs 4
Subject 01, session 05, accumulated dfs 5
Subject 01, session 06, accumulated dfs 6
Subject 01, session 07, accumulated dfs 7
Subject 01, session 08, accumulated dfs 8
Subject 01, session 09, accumulated dfs 9
Subject 01, session 10, accumulated dfs 10
Failed: ../data/fmriprep_output/derivatives/sub-MSC01/ses-func11/datasets/sub-MSC01_ses-func11_task-memoryscenes_roi-rnd_r_encoding-stimxhrf_desc-vox_w_stim_hrf.csv
Failed: ../data/fmriprep_output/derivatives/sub-MSC01/ses-func12/datasets/sub-MSC01_ses-func12_task-memoryscenes_roi-rnd_r_encoding-stimxhrf_desc-vox_w_stim_hrf.csv
Failed: ../data/fmriprep_output/derivatives/sub-MSC01/ses-func13/datasets/sub-MSC01_ses-func13_task-memoryscenes_roi-rnd_r_encoding-stimxhrf_desc-vox_w_stim_hrf.csv
Failed: ../data/fmriprep_output/derivatives/sub-MSC01/ses-func14/datasets/sub-MSC01_

In [68]:
save_dataset_with_metadata(fdf)

2026-01-02 20:59:39 - INFO - Successfully saved unique dataset and metadata to: datasets/dataset_task-memoryscenes_roi-rnd_r
